# 🧪 AiStock 数据加载服务测试

## 测试目标
- ✅ DataLoader: 统一数据加载接口
- ✅ TDXAdapter: TDX 行情接口
- ✅ ExternalAPI: 外盘期货数据

## 测试数据
- A 股日线数据
- 宏观指标数据
- 外盘期货价格

In [ ]:
# 环境设置
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成")

In [ ]:
# 导入数据服务
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from dynamic_price_system.data.data_loader import DataLoader
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

print("✅ 数据服务导入成功")

In [ ]:
# 初始化服务
config = ConfigService(system_name='dynamic_price')
cache = CacheService(max_size=1000, ttl=3600)
db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)

data_loader = DataLoader(
    config_service=config,
    cache_service=cache,
    db_main=db,
    enable_cache=True
)

print("✅ 数据加载服务初始化成功")

## 1️⃣ 股票日线数据测试

In [ ]:
# 测试单只股票数据加载
test_stocks = ['600938', '601899', '300750']

print("📊 测试股票数据加载：")
for code in test_stocks:
    df = data_loader.load_stock_daily(code, min_days=100)
    if df is not None and len(df) > 0:
        print(f"  ✅ {code}: {len(df)}条数据 | 最新价：{df['close'].iloc[-1]:.2f}")
    else:
        print(f"  ❌ {code}: 数据加载失败")

In [ ]:
# 可视化股票数据
df = data_loader.load_stock_daily('600938', min_days=100)

if df is not None and len(df) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # 收盘价
    axes[0].plot(df['datetime'], df['close'], label='收盘价', linewidth=2)
    axes[0].set_title('中国海油 (600938) 收盘价')
    axes[0].set_xlabel('日期')
    axes[0].set_ylabel('价格 (元)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 成交量
    axes[1].bar(df['datetime'], df['volume'], label='成交量', alpha=0.7)
    axes[1].set_title('成交量')
    axes[1].set_xlabel('日期')
    axes[1].set_ylabel('成交量')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("✅ 股票数据可视化完成")

## 2️⃣ 宏观指标数据测试

In [ ]:
# 测试宏观指标加载
macro_data = data_loader.load_all_macro()

print("\n📊 宏观指标数据：")
for name, value in macro_data.items():
    if value is not None:
        print(f"  ✅ {name}: {value}")
    else:
        print(f"  ⚠️ {name}: 数据不可用")

print(f"\n✅ 共加载 {len(macro_data)} 个宏观指标")

## 3️⃣ 财务数据测试

In [ ]:
# 测试财务数据加载
financial_data = data_loader.load_all_financial()

print("\n📊 财务数据测试：")
for code, data in list(financial_data.items())[:5]:
    print(f"\n  {code}:")
    for k, v in data.items():
        if v is not None:
            print(f"    • {k}: {v}")

## 4️⃣ 缓存性能测试

In [ ]:
# 测试缓存性能
import time

print("\n📊 缓存性能测试：")

# 首次加载（缓存未命中）
start = time.time()
df1 = data_loader.load_stock_daily('600938', min_days=100)
time1 = time.time() - start

# 二次加载（缓存命中）
start = time.time()
df2 = data_loader.load_stock_daily('600938', min_days=100)
time2 = time.time() - start

print(f"  • 首次加载（未命中）: {time1:.3f}秒")
print(f"  • 二次加载（命中）: {time2:.3f}秒")
print(f"  • 性能提升：{(time1/time2):.1f}x")

# 缓存统计
stats = data_loader.get_cache_stats()
print(f"\n📊 缓存统计：")
print(f"  • 命中率：{stats['hit_rate']:.1%}")
print(f"  • 命中/未命中：{stats['hits']}/{stats['misses']}")
print(f"  • 当前容量：{stats['current_size']}/{stats['max_size']}")

## 📊 测试总结

In [ ]:
print("="*60)
print("📋 数据加载服务测试总结")
print("="*60)
print(f"✅ 股票数据：{len(test_stocks)}只测试成功")
print(f"✅ 宏观指标：{len(macro_data)}个加载成功")
print(f"✅ 财务数据：{len(financial_data)}只加载成功")
print(f"✅ 缓存命中率：{stats['hit_rate']:.1%}")
print("="*60)

# 清理资源
db.close()
cache.clear()
print("\n✅ 资源已清理")